# 06 — Diagnostics (Phase D-1)

FF5 + UMD neutrality regression on the headline TCNN rungs (5w, 6w) plus a hand-crafted baseline (rung_2) for comparison. The intercept α is the residual return after controlling for the systematic factors that explain most cross-sectional returns. The question this notebook answers:

> Is the rung_6w 0.66 gross Sharpe **residual alpha after FF5+UMD**, or just **momentum-factor beta dressed up**?

**Pass criteria (two-axis):**
- **Strict pass**: α t-stat ≥ 2.0 AND all factor betas |β| < 0.05
- **Soft pass**: factor-orthogonal (all β small, R² < 1%) even if α t-stat < 2.0 — defensible as a small but novel signal
- **Fail**: significant factor loading (especially UMD) → strategy is replicating a known factor

In [1]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src import config, eval as evalmod

## 1. Load FF5+UMD factor data and remap strategy returns to trading-day calendar

**Important**: `all_results.csv` returns are labeled with `rebal_date + Timedelta(days=di+1)` (calendar days), but FF5+UMD is only on trading days. Without remapping, ~32% of returns silently drop in the merge. We rebuild the date labels from the panel's actual trading-day calendar.

FF5+UMD merged data: `03_UCHICAGO_BOOTH/Retail_Sentiment/data/processed/ff5_umd_daily.parquet` (run `python scripts/pull_umd_from_wrds.py` once if missing).

In [2]:
FF_UMD_PATH = Path('..').resolve().parent.parent / '03_UCHICAGO_BOOTH' / 'Retail_Sentiment' / 'data' / 'processed' / 'ff5_umd_daily.parquet'
assert FF_UMD_PATH.exists(), f'Missing: {FF_UMD_PATH}\nRun: python scripts/pull_umd_from_wrds.py'
ff5umd = pd.read_parquet(FF_UMD_PATH)
ff5umd['date'] = pd.to_datetime(ff5umd['date'])
print(f'ff5+umd: {len(ff5umd):,} rows, {ff5umd["date"].min().date()} -> {ff5umd["date"].max().date()}')
print(f'columns: {list(ff5umd.columns)}')

trading_days = pd.DatetimeIndex(sorted(pd.to_datetime(
    pd.read_parquet(config.PANEL_DAILY_PARQUET, columns=['date'])['date']
).unique()))
print(f'panel trading days: {len(trading_days):,}')

def remap_to_trading_days(master_df):
    out = []
    for (exp_id, rebal), group in master_df.groupby(['experiment_id','rebal_date'], sort=True):
        g = group.sort_values('date').copy()
        n = len(g)
        idx = trading_days.searchsorted(rebal, side='right')
        new_dates = trading_days[idx: idx + n]
        if len(new_dates) < n:
            new_dates = list(new_dates) + [pd.NaT] * (n - len(new_dates))
        g['date'] = list(new_dates)
        out.append(g)
    return pd.concat(out, ignore_index=True).dropna(subset=['date'])

ff5+umd: 15,770 rows, 1963-07-01 -> 2026-02-27
columns: ['date', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf', 'umd']
panel trading days: 8,817


## 2. FF5+UMD regression on TCNN rungs + baseline

In [3]:
TARGETS = ['rung_2_ewm_momentum_decile',
           'rung_5_tcnn_1ch', 'rung_5w_tcnn_1ch_winsor',
           'rung_6_tcnn_3ch', 'rung_6w_tcnn_3ch_winsor']
master = evalmod.load_master_results(TARGETS)
master['date'] = pd.to_datetime(master['date'])
master['rebal_date'] = pd.to_datetime(master['rebal_date'])
master = remap_to_trading_days(master)

rows = []
for exp_id in TARGETS:
    df = master[master['experiment_id'] == exp_id].sort_values('date').drop_duplicates('date').set_index('date')
    res = evalmod.ff5_neutrality(df['return'], ff5umd, include_umd=True)
    row = {'rung': exp_id, 'alpha_ann_pct': res['alpha_ann']*100, 'alpha_t': res['alpha_t'],
           'R2': res['r_squared'], 'n_obs': res['n_obs']}
    row.update({f'b_{k}': v for k, v in res['betas'].items()})
    rows.append(row)

results = pd.DataFrame(rows).set_index('rung')
print('=== FF5+UMD neutrality regression (trading-day remapped) ===')
print(results.round(3).to_string())

=== FF5+UMD neutrality regression (trading-day remapped) ===
                            alpha_ann_pct  alpha_t     R2  n_obs  b_mktrf  b_smb  b_hml  b_rmw  b_cma  b_umd
rung                                                                                                        
rung_2_ewm_momentum_decile          3.232    1.065  0.006   3086   -0.031  0.034 -0.017 -0.027 -0.001  0.033
rung_5_tcnn_1ch                     4.521    0.653  0.001   3336   -0.009  0.093 -0.013  0.090  0.001 -0.003
rung_5w_tcnn_1ch_winsor             3.452    1.308  0.001   3336   -0.002  0.011  0.011  0.036 -0.049  0.004
rung_6_tcnn_3ch                     0.852    0.863  0.007   3336   -0.009  0.013  0.014  0.013 -0.031 -0.008
rung_6w_tcnn_3ch_winsor             1.349    1.495  0.005   3336   -0.004  0.014  0.006  0.011 -0.017 -0.006


## 3. D-1 gate

In [4]:
HEADLINE = 'rung_6w_tcnn_3ch_winsor'
alpha_t = float(results.loc[HEADLINE, 'alpha_t'])
alpha_ann = float(results.loc[HEADLINE, 'alpha_ann_pct'])
r2 = float(results.loc[HEADLINE, 'R2'])
betas = {c.replace('b_',''): float(results.loc[HEADLINE, c]) for c in results.columns if c.startswith('b_')}
max_beta_abs = max(abs(v) for v in betas.values())

print(f'\n{HEADLINE}:')
print(f'  Annualized alpha:        {alpha_ann:+.2f}%')
print(f'  Alpha t-statistic:       {alpha_t:+.2f}  (strict gate: >= 2.0)')
print(f'  R-squared:               {r2:.4f}  (factors explain {r2*100:.2f}% of variance)')
print(f'  Max |beta| across FF5+UMD: {max_beta_abs:.3f}')
print(f'  UMD beta:                {betas.get("umd", float("nan")):+.4f}')
print(f'  MKT beta:                {betas.get("mktrf", float("nan")):+.4f}')

orthogonal = (max_beta_abs < 0.05) and (r2 < 0.02)
print()
if alpha_t >= 2.0 and orthogonal:
    print('D-1 STRICT PASS \u2014 significant alpha AND factor-orthogonal.')
elif orthogonal:
    print('D-1 SOFT PASS \u2014 strategy is factor-orthogonal (all betas tiny, R\u00b2 < 1%) but')
    print('              absolute residual alpha is small enough that t-stat < 2.0.')
    print('              Defensible framing: this is NOT momentum beta. The signal is')
    print('              a factor-orthogonal cross-sectional bet. More seeds or a longer')
    print('              sample would tighten the t-stat without changing the orthogonality.')
elif alpha_t >= 2.0:
    print('D-1 PARTIAL \u2014 significant alpha but some factor loading. Disentangle the source.')
else:
    print('D-1 FAIL \u2014 alpha not significant AND non-trivial factor exposure. Reframe pitch.')


rung_6w_tcnn_3ch_winsor:
  Annualized alpha:        +1.35%
  Alpha t-statistic:       +1.50  (strict gate: >= 2.0)
  R-squared:               0.0045  (factors explain 0.45% of variance)
  Max |beta| across FF5+UMD: 0.017
  UMD beta:                -0.0060
  MKT beta:                -0.0038

D-1 SOFT PASS — strategy is factor-orthogonal (all betas tiny, R² < 1%) but
              absolute residual alpha is small enough that t-stat < 2.0.
              Defensible framing: this is NOT momentum beta. The signal is
              a factor-orthogonal cross-sectional bet. More seeds or a longer
              sample would tighten the t-stat without changing the orthogonality.
